# Fine-tune Amazon Nova with Amazon Bedrock

This notebook demonstrates how to fine-tune Amazon Nova using Amazon Bedrock. The process includes:

1. Setting up model and training configurations
2. Configuring SageMaker resources
3. Fine-tuning the model
4. Evaluating the fine-tuning training process
5. Downloading and analyzing the fine-tuned model

## Key Components

- Model Configuration: Select and configure the model to be fine-tuned
- SageMaker Setup: Configure AWS resources and training environment
- Training Process: Fine-tune the model using the SWIFT framework
- Evaluation: Analyze training metrics and model performance
- Model Export: Save and prepare the model for deployment

## Requirements

- AWS SageMaker access with appropriate permissions
- Training data in the correct format
- Sufficient GPU resources for training

In [ ]:
%pip install --quiet -U boto3 botocore pathvalidate

In [ ]:
import boto3 
from botocore.config import Config
import sys
import pandas as pd
import matplotlib.pyplot as plt
import json
import time 
import concurrent.futures
import tqdm
import os
import uuid
import ipywidgets as widgets
from IPython.display import display
from pathvalidate import sanitize_filename
from utils.bedrock import get_fine_tunable_vision_models

In [ ]:
import sagemaker

In [ ]:
request_uuid = uuid.uuid4()

In [ ]:
region="us-east-1" # us-east-1 for Amazon Nova models
boto_session = boto3.Session(
    region_name=region
)
session = sagemaker.Session(boto_session=boto_session)
default_bucket_name = session.default_bucket() # "nova-doc-to-json-w2" 
dataset_s3_prefix = "fatura2-train-data-amazon-nova-v3-3-300" # "fatura2-train-data-amazon-nova-v2-1k" # "fatura2-train-data-amazon-nova-v2-3-300" # "w2-train-data-amazon-nova" 
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"

In [ ]:
account_id = session.account_id()

In [ ]:
dataset_s3_uri

In [ ]:
my_config = Config(
    region_name = region, 
    signature_version = 'v4',
    retries = {
        'max_attempts': 5,
        'mode': 'standard'
    })

bedrock = boto_session.client(service_name="bedrock", config=my_config)

In [ ]:
def get_model_arn_by_id_from(fine_tunable_models, model_id):
    return next((model["modelArn"] for model in fine_tunable_models if model.get("modelId","") == model_id)) 

In [ ]:
fine_tunable_models = get_fine_tunable_vision_models(bedrock)

In [ ]:
global model_id
global model_arn
global safe_model_id
model_id = "amazon.nova-lite-v1:0:300k"
model_arn = get_model_arn_by_id_from(fine_tunable_models, model_id)
safe_model_id = sanitize_filename(model_id)

In [ ]:
model_dropdown = widgets.Dropdown(
    options=[model["modelId"] for model in fine_tunable_models],
    value=model_id,
    description='Model:',
    style={'description_width': 'initial'},
    layout={'width': 'auto'}
)

# Create select button
select_button = widgets.Button(
    description='Select Model',
    style={'description_width': 'initial'},
    button_style='success',
    layout={'width': 'auto'}
)

# Create output widget for status messages
output = widgets.Output()

def on_select_button_click(b):
    global model_id
    global model_arn
    global safe_model_id
    with output:
        output.clear_output()
        model_id = model_dropdown.value
        model_arn = get_model_arn_by_id_from(fine_tunable_models, model_id)
        safe_model_id = sanitize_filename(model_id)
        print(f"Selected model: {model_id}")
        print(f"Model ARN: {model_arn}")
        print(f"Path safe model id: {model_id}")

select_button.on_click(on_select_button_click)

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Model to Fine-Tune</h3>"),
    model_dropdown,
    select_button,
    output
]))


In [ ]:
from datetime import datetime

now = datetime.now()
formatted_time = now.strftime('%Y-%m-%d-%H-%M-%S') + "-" + now.strftime('%f')[:3] # Output: 2025-05-07-14-43-xx-xxx

In [ ]:
## Specify input S3 bucket
input_s3_uri = os.path.join(dataset_s3_uri,"conversations_train_nova_format.jsonl")
validation_s3_uri = os.path.join(dataset_s3_uri,"conversations_dev_nova_format.jsonl")
output_s3_uri = f"s3://{default_bucket_name}/bedrock-finetune-output/{safe_model_id}/{formatted_time}/"

## Create the Amazon Bedrock Customization Service Role

In [ ]:
# Create IAM client
iam_client = session.boto_session.client('iam')

# Define the role name
role_name = "AmazonBedrockFineTuneServiceRole"

# Define the policy document
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                f"arn:aws:s3:::{default_bucket_name}",
                f"arn:aws:s3:::{default_bucket_name}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
            ],
            "Resource": [
                f"arn:aws:s3:::{default_bucket_name}/bedrock-finetune-output/*"
            ]
        }
    ]
}

# Define the trust relationship
trust_relationship = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock.amazonaws.com"
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "aws:SourceAccount": account_id
                },
                "ArnEquals": {
                    "aws:SourceArn": f"arn:aws:bedrock:{my_config.region_name}:{account_id}:model-customization-job/*"
                }
            }
        }
    ]
}



# Check if the role already exists
try:
    existing_role = iam_client.get_role(RoleName=role_name)
    print(f"Role '{role_name}' already exists. Skipping creation.")
    roleArn = existing_role['Role']['Arn']
    # TODO if role exists check that it has the necessary permissions. if not add permissions. 
except iam_client.exceptions.NoSuchEntityException:
    # Create the role
    try:
        create_role_response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_relationship),
            Description="Service role for Amazon Bedrock Fine-Tuning"
        )
        
        # Attach the inline policy to the role
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName="AmazonBedrockCustomizationServiceRolePolicy",
            PolicyDocument=json.dumps(policy_document)
        )
        
        print(f"Successfully created role: {create_role_response['Role']['Arn']}")
        roleArn = create_role_response['Role']['Arn']
    except Exception as e:
        print(f"Error creating role: {str(e)}")



In [ ]:
# Nova model customization currently only available in US EAST 1
role_arn = roleArn

CUSTOM_MODEL_NAME_MAX_LENGTH = 63
job_name = f"-{formatted_time}"
# dataset_s3_prefix = dataset_s3_prefix.replace("v6","v7-epoch-5")
job_name = f"{dataset_s3_prefix[:(CUSTOM_MODEL_NAME_MAX_LENGTH-len(job_name)-1)]}{job_name}"
model_name = job_name 

In [ ]:
validation_s3_uri

In [ ]:
# Select the customization type from "FINE_TUNING" or "CONTINUED_PRE_TRAINING". 
customization_type = "FINE_TUNING"


# Define the hyperparameters for fine-tuning Amazon Nova model
hyper_parameters = {
    "epochCount": "5", # default: 2
    "learningRate": '0.00001', 
    "batchSize": "1",
    "learningRateWarmupSteps": '2' # Nova Lite: (dataset size / 160), Nova Pro: (dataset size / 320)
}


response_ft = bedrock.create_model_customization_job(
    customizationType=customization_type,
    clientRequestToken=str(request_uuid),
    jobName = job_name,
    customModelName = model_name,
    roleArn = role_arn,
    baseModelIdentifier = model_arn,
    hyperParameters=hyper_parameters,
    trainingDataConfig={"s3Uri": input_s3_uri},
    validationDataConfig={
        "validators": [
            {"s3Uri": validation_s3_uri},
        ]
    },
    outputDataConfig={"s3Uri": output_s3_uri},
)


In [ ]:
response_ft

## Check fine-tuning job status

In [ ]:
jobArn = response_ft.get('jobArn')

In [ ]:
model_name = "fatura2-train-data-amazon-nova-2025-10-14-14-29-45-441" # "fatura2-train-data-amazon-nova-2025-05-20-15-06-06-468"
job_name = model_name

In [ ]:

status = bedrock.get_model_customization_job(jobIdentifier=jobArn)["status"]
print(f'Job status: {status}')

## Deploy fine-tuned Amazon Nova with provisioned throughput

In [ ]:
provisioned_model_id = bedrock.create_provisioned_model_throughput(
    modelUnits=1,
    provisionedModelName=model_name,
    modelId=job_name,
    # commitmentDuration='OneMonth'|'SixMonths', # Omit this field to request a no-commit Provisioned Throughput
)

print(provisioned_model_id['provisionedModelArn'])

In [ ]:
provisioned_model_id

In [ ]:
provisioning = True
while provisioning:
    status_provisioning = bedrock.get_provisioned_model_throughput(provisionedModelId = provisioned_model_id['provisionedModelArn'])['status']
    provisioning = status_provisioning == 'Creating'
    print(f"{status_provisioning}..." if provisioning else status_provisioning)
    time.sleep(120)

## Deploy fine-tuned Amazon Nova Model with on-demand inference

In [ ]:
status

In [ ]:




# If job completed successfully, get the model details and create deployment
if status == "Completed":
    # Get custom model ARN
    model_details = bedrock.get_model_customization_job(jobIdentifier=jobArn)
    custom_model_arn = model_details["outputModelArn"]
    print(f"Fine-tuned model ARN: {custom_model_arn}")
    
    # Create on-demand deployment
    deployment_arn = create_model_deployment(custom_model_arn)
    
    if deployment_arn:
        # Check initial status
        initial_status = check_deployment_status(deployment_arn)
        
        # Store the deployment ARN for later use
        %store deployment_arn



Wait for Deployment to Complete

ℹ️ Info: It takes about 30 mins to complete the deployment

Let's monitor our deployment until it's ready for use:

In [ ]:


# Function to wait for deployment to be ready
def wait_for_deployment(deployment_arn, max_wait_seconds=3600, check_interval=60):
    """
    Wait for a deployment to reach 'IN_SERVICE' status
    
    Parameters:
    -----------
    deployment_arn : str
        ARN of the deployment to monitor
    max_wait_seconds : int
        Maximum wait time in seconds (default: 1800s = 30 minutes)
    check_interval : int
        Interval between checks in seconds (default: 60s)
        
    Returns:
    --------
    success : bool
        True if deployment is in service, False otherwise
    """
    import time
    from tqdm.notebook import tqdm
    
    start_time = time.time()
    end_time = start_time + max_wait_seconds
    
    print(f"Waiting for deployment to complete (max wait time: {max_wait_seconds/60:.1f} minutes)")
    
    # Create a progress bar for the wait time
    with tqdm(total=max_wait_seconds, desc="Waiting for deployment", unit="sec") as pbar:
        elapsed = 0
        while time.time() < end_time:
            status = check_deployment_status(deployment_arn)
            
            if status == "Active":
                print(f"\n✅ Deployment is now in service after {(time.time() - start_time)/60:.1f} minutes")
                return True
                
            if status == "Failed":
                print(f"\n❌ Deployment failed or was deleted. Final status: {status}")
                return False
                
            # Update progress bar with time elapsed since last check
            new_elapsed = int(time.time() - start_time)
            pbar.update(new_elapsed - elapsed)
            elapsed = new_elapsed
            
            # Wait before checking again
            time.sleep(check_interval)
    
    print(f"\n⚠️ Timed out after waiting {max_wait_seconds/60:.1f} minutes. Deployment may still be in progress.")
    return False

# Only attempt to wait for deployment if it was created
if 'deployment_arn' in locals() and deployment_arn:
    deployment_ready = wait_for_deployment(deployment_arn)
    
    if deployment_ready:
        %store deployment_arn
    else:
        print("Deployment did not complete successfully. Using the base custom model for inference.")



## Batch Inference for Evaluation

In [ ]:
%pip install --upgrade parallel-pandas --quiet

In [ ]:
import boto3
import sagemaker
import os
import pandas as pd
import json
from parallel_pandas import ParallelPandas

In [ ]:
region="us-east-1" # us-east-1 for Amazon Nova models
boto_session = boto3.Session(region_name=region)

# Initialize AWS resources
session = sagemaker.Session(boto_session=boto_session)
# default_bucket_name = "nova-doc-to-json-905418197933" # session.default_bucket()
# dataset_s3_prefix = "v6-3x300-fatura2-train-data-amazon-nova" # "fatura2-train-data-amazon-nova"
default_bucket_name = session.default_bucket() # "nova-doc-to-json-w2" 
dataset_s3_prefix = "fatura2-train-data-amazon-nova" # "w2-train-data-amazon-nova" 
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"

# Create local directory structure
data_main_dir = "./data/"
hf_dataset_name = "arlind0xbb/Fatura2-invoices-original-strat2"
# hf_dataset_name = "singhsays/fake-w2-us-tax-form-dataset"
dataset_dir = os.path.join(data_main_dir, hf_dataset_name.split("/")[-1])
os.makedirs(dataset_dir, exist_ok=True)

results_dir = "./data/results"
# TODO use model name or training job name
output_dir = "fatura2"
output_dir_path = os.path.join(results_dir, output_dir)
os.makedirs(output_dir_path, exist_ok=True)

In [ ]:
model_id = deployment_arn

In [ ]:
model_id = "us.amazon.nova-lite-v1:0"

In [ ]:
!aws s3 sync $dataset_s3_uri $dataset_dir --quiet

In [ ]:
test_data_file = "conversations_test_nova_format.jsonl"

In [ ]:
ParallelPandas.initialize(n_cpu=2) # 2 request is parallel seems to be the limit for 1 MU

In [ ]:
df = pd.read_json(os.path.join(dataset_dir,test_data_file), lines=True)

In [ ]:
def pop_assistant_ground_truth(messages):
    # remove last assistant turn
    label = None
    if messages[-1]['role'] == "assistant":
        assistant = messages.pop()
        # Extract ground truth
        label = assistant["content"][0]["text"] # TODO consider adding label to initial dataset
    
    return (messages, label)
    
df[['messages', 'labels']] = pd.DataFrame(df['messages'].apply(lambda messages: pop_assistant_ground_truth(messages)).tolist(), index=df.index)


In [ ]:
def get_local_image_location(messages):
    user_msg = messages[0]
    images = [ for msg in user_msg]
    # TODO

df["images"] = df['messages'].apply()

In [ ]:
df.iloc[0]["labels"]

In [ ]:
df.iloc[0]["messages"]

In [ ]:
# Configure AWS client 
bedrock_rt_client = boto_session.client(
    service_name='bedrock-runtime',
    region_name=region,
)

In [ ]:
# API setting constants
API_MAX_RETRY = 16
API_RETRY_SLEEP = 10
API_ERROR_OUTPUT = "$ERROR$"


def chat_completion_aws_bedrock_nova(model, messages, temperature, max_tokens,bedrock_rt_client = None ):  
    if not bedrock_rt_client:
        bedrock_rt_client = boto3.client('bedrock-runtime')

    output = "{}"
    # Retry logic for API calls
    for _ in range(API_MAX_RETRY):
        try:
            # Create messages from conversation
            inferenceConfig = {
                "max_new_tokens": max_tokens,
                "temperature": temperature, 
            }

            # Prepare request body
            model_kwargs = {"messages": messages,
                            "inferenceConfig": inferenceConfig}
            body = json.dumps(model_kwargs)

            # Call Bedrock API
            response = bedrock_rt_client.invoke_model(
                body=body,
                modelId=model,
                accept='application/json',
                contentType='application/json'
            )

            # Parse response
            response_body = json.loads(response.get('body').read())
            
            output = response_body['output']['message']['content'][0]['text']
            break

        except Exception as e:
            print(type(e), e)
            return str(e)
            ## Uncomment time.sleep if encounter Bedrock invoke throttling error
            # time.sleep(API_RETRY_SLEEP)

    return output

In [ ]:
# API setting constants
API_MAX_RETRY = 16
API_RETRY_SLEEP = 10
API_ERROR_OUTPUT = "$ERROR$"


def chat_completion_aws_bedrock_nova_converse(model, messages, temperature, max_tokens, bedrock_rt_client = None, tool_config = None ):  
    if not bedrock_rt_client:
        bedrock_rt_client = boto3.client('bedrock-runtime')

    output = "{}"
    # Retry logic for API calls
    for _ in range(API_MAX_RETRY):
        try:
            # Create messages from conversation
            inferenceConfig = {
                "maxTokens": max_tokens,
                "temperature": temperature, 
            }


            model_response = bedrock_rt_client.converse(
                modelId=model, 
                messages=messages, 
                inferenceConfig=inferenceConfig,
                toolConfig=tool_config
            )
            
            output = model_response["output"]["message"]["content"][0]["text"]
            # messages.append(model_response["output"]["message"])

            # Pretty print the response JSON.
            print("[Full Response]")
            print(json.dumps(model_response, indent=2))
            
            # Print the tool content for easy readability.
            tool = next(
                block["toolUse"]
                for block in model_response["output"]["message"]["content"]
                if "toolUse" in block
            )
            print("\n[Tool Response]")
            print(tool)
            break

        except Exception as e:
            print(type(e), e)
            return str(e)
            ## Uncomment time.sleep if encounter Bedrock invoke throttling error
            # time.sleep(API_RETRY_SLEEP)

    return output

In [ ]:
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "structured_output", # Name of the tool
                "description": "JSON to extract from document", # Concise description of the tool
                "inputSchema": {
                    "json": { 
                        "type": "object",
                        "properties": {
                            "AMOUNT_DUE": {
                              "type": [
                                "null",
                                "string"
                              ]
                            }
                          },
                          "required": [
                            "AMOUNT_DUE"
                          ]
                    }
                }
            }
        }
    ]
}

In [ ]:
# user_query = "10*5"

# messages = [{
#     "role": "user",
#     "content": [{"text": user_query}]
# }]

tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "document_output", # Name of the tool
                "description": "Tool to extract information from the document", # Concise description of the tool
                "inputSchema": {
                    "json": { 
                        "type": "object",
                        "properties": {
                            "amount_due": { # The name of the parameter
                                "type": "string", # parameter type: string/int/etc
                                #"description": "The monetary amount on the invoice" # Helpful description of the parameter
                            }
                        },
                        "required": [ # List of all required parameters
                            "amount_due"
                        ]
                    }
                }
            }
        }
    ]
}

In [ ]:
df.iloc[0]["messages"]

In [ ]:
converse_out = chat_completion_aws_bedrock_nova_converse(model_id, df.iloc[0]["messages"], 0, 1024, bedrock_rt_client, tool_config)

In [ ]:
converse_out = chat_completion_aws_bedrock_nova_converse(model_id, df.iloc[0]["messages"], 0, 1024, bedrock_rt_client, tool_config)

In [ ]:
output = chat_completion_aws_bedrock_nova(model_id, df.iloc[0]["messages"], 0, 1024, bedrock_rt_client)

In [ ]:
output

In [ ]:
temperature = 0
max_tokens = 4096

In [ ]:
df['response'] = df.p_apply(lambda document_row: chat_completion_aws_bedrock_nova(model_id, document_row["messages"], temperature, max_tokens, bedrock_rt_client), axis=1)

In [ ]:
errors = df[df['response'].apply(lambda x: "error" in x)]

In [ ]:
len(errors)

In [ ]:

errors['response'] = errors.p_apply(lambda document_row: chat_completion_aws_bedrock_nova(model_id, document_row["messages"], temperature, max_tokens, bedrock_rt_client), axis=1)

In [ ]:
df.update(errors)

In [ ]:
output_dir_path

In [ ]:
df.to_json(os.path.join(output_dir_path,"results.jsonl"), orient='records', lines=True)

In [ ]:
output_dir

In [ ]:
results_archive = os.path.join(results_dir, output_dir + ".tar.gz")

In [ ]:
import tarfile
import os

def create_tar_gz(output_filename, source_dir):
    with tarfile.open(output_filename, "w:gz") as tar:
        tar.add(source_dir, arcname=os.path.basename(source_dir))
        
# Example usage
create_tar_gz(results_archive, output_dir_path)

In [ ]:
results_archive

In [ ]:
dataset_s3_prefix

In [ ]:
!aws s3 cp $results_archive "s3://<BUCKET_>/v9-nova-300-us-east-1-2025-10-13-13-31-47-871/fatura2-results.tar.gz"

In [ ]:
# todo remove hard-coded s3
!aws s3 cp './data/results/v8-factura-fr.tar.gz' "s3://<BUCKET>/v8-factura-fr.tar.gz"


In [ ]:
# add to tracking csv

## Optional Plot training loss
Optionally, you can also plot training loss using the step_wise_training_metrics.csv file generated from the finetuning job. This csv file and other model artifacts can be found under Amazon Bedrock -> Custom model -> Custom model name -> Output data (S3 location)

In [ ]:

def plot_training_loss(input_file, output_file):
    ''' This function plots training loss using the default model output file 'step_wise_training_metrics.csv' generated from the finetuning job'''
    
    # Read the CSV file
    df = pd.read_csv(input_file)
    
    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.plot(df['step_number'], df['training_loss'], 'b-', linewidth=2)
    
    # Customize the plot
    plt.title('Training Loss vs Step Number', fontsize=14)
    plt.xlabel('Step Number', fontsize=12)
    plt.ylabel('Training Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Add some padding to the axes
    plt.margins(x=0.02)
    
    # Save the plot
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Plot saved as {output_file}")


# Example usage

plot_training_loss(input_file = 'step_wise_training_metrics.csv', #'model_training_loss/aws-ft-nova-lite/step_wise_training_metrics_epoch5_lr_1e-06.csv', 
                   output_file = 'step_wise_training_metrics.png') # 'model_training_loss/aws-ft-nova-lite/training_loss_epoch5_lr_1e-06.png')